In [ ]:
# Load R magic extension for Python Jupyter kernel (RData loading)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 5-Class Emergency Department Triage Classifier: FedMML Training, 5V-Adapted Class Weighting & Benchmark Comparison (`models/train_fedmm_classifier.ipynb`)

This notebook implements and benchmarks two training paradigms for **LightGBM Emergency Severity Index (ESI 1..5) triage models** on the **Federated Multi-Modal Emergency Dataset (`datasets/fedmml_ed_triage_dataset.csv`)** using 6 core vital signs and demographic features, evaluated on both internal FedMML test data and the **5-Variable MIMIC Dataset (`datasets/5v_cleandf.RData`)**:

### 🔬 Two Comparative Benchmarks
1. **Benchmark 1 (Standard FedMML Balanced Weighting)**:
   - Trains Direct Multiclass & Hierarchical LightGBM models using balanced class weighting derived from FedMML training data.
2. **Benchmark 2 (5V-Adapted Class Weighting)**:
   - Copies the exact model architectures trained on FedMML predictor features, but adjusts the **loss class weights** according to the global empirical class distribution of the entire 5V dataset ($N=282,558$ complete cases):
     $$w_k = \frac{N_{5V}}{5 \times N_{5V, k}}$$

### 📊 Feature Roster & Cross-Dataset Translation
| Feature / Target | FedMML Dataset (`fedmml_ed_triage_dataset.csv`) | 5V Dataset (`5v_cleandf.RData`) | Mapping / Encoding Logic |
| :--- | :--- | :--- | :--- |
| **Age** | `age` | `age` | Continuous numerical (years) |
| **Sex / Gender** | `sex` (`'M'`, `'F'`) | `gender` (`'Male'`, `'Female'`) | Binary integer: `1 = Male`, `0 = Female` |
| **Systolic BP** | `systolic_bp` | `triage_vital_sbp` | Continuous numerical (mmHg) |
| **Heart Rate** | `heart_rate` | `triage_vital_hr` | Continuous numerical (bpm) |
| **Respiratory Rate** | `respiratory_rate` | `triage_vital_rr` | Continuous numerical (breaths/min) |
| **Oxygen Saturation** | `spo2` | `triage_vital_o2` | Continuous numerical (%) |
| **Target Class** | `esi_level` (`1, 2, 3, 4, 5`) | `esi` (`'1', '2', '3', '4', '5'`) | Discrete 5-class target ($1..5$) |

### 🧹 Complete Case Cleaning Rules
- **FedMML**: Strictly drops rows with $\ge 1$ null value in `['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2', 'esi_level']` (retains **81,742 complete cases**).
- **5V Dataset**: Strictly skips and drops rows with $\ge 1$ null value in `['age', 'gender', 'triage_vital_sbp', 'triage_vital_hr', 'triage_vital_rr', 'triage_vital_o2', 'esi']` (retains **282,558 complete cases**).

```mermaid
flowchart TD
    FedMML["FedMML Dataset (81,742 complete rows)"] --> Split["Stratified 3-Way Split: Train 70%, Val 20%, Test 10%"]
    Split --> Preproc["StandardScaler Normalization"]
    
    Dataset5V["External 5V Dataset (282,558 complete rows)"] --> Weights5V["Compute Global 5V Target Class Distribution & Weights"]
    
    Preproc --> TrainBench1["Benchmark 1: Standard FedMML Balanced Weighting (Multiclass & Stacking)"]
    Preproc & Weights5V --> TrainBench2["Benchmark 2: 5V-Adapted Class Weighting (Multiclass & Stacking)"]
    
    TrainBench1 & TrainBench2 --> EvalFedMML["Evaluation on Internal FedMML Holdout Test Set (8,175 rows)"]
    TrainBench1 & TrainBench2 --> Eval5V["Evaluation on External 5V Generalization Dataset (282,558 rows)"]
    
    EvalFedMML & Eval5V --> FinalComp["Comparative Analysis: Standard vs 5V-Adapted Class Weighting"]
```

In [ ]:
# ---------------------------------------------------------
# Step 1: Load FedMML Dataset, Filter Nulls, Encode 'sex', & Stratified 3-Way Split via triage_conf.json
# ---------------------------------------------------------
import os, json, pickle, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

# 1. Load Partitioning Configuration
config_path = f"{ROOT}/config/triage_conf.json"
with open(config_path, 'r') as f:
    config = json.load(f)

test_size = config['training']['test_size']
val_size  = config['training']['val_size']
seed_val  = config['training']['random_state']

print(f"Loaded Configuration from {config_path}:")
print(f"  * Test Size Fraction       = {test_size:.2f} ({test_size*100:.1f}%)")
print(f"  * Validation Size Fraction = {val_size:.2f} ({val_size*100:.1f}%)")
print(f"  * Random State Seed        = {seed_val}")
print("-" * 80)

# 2. Load Raw FedMML Dataset
data_path = f"{ROOT}/datasets/fedmml_ed_triage_dataset.csv"
print(f"Loading FedMML dataset from: {data_path}...")
raw_df = pd.read_csv(data_path)
initial_rows = len(raw_df)

required_features = ['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']
target_col = 'esi_level'
all_required_cols = required_features + [target_col]

print("=" * 80)
print(f"  FEDMML DATASET: {initial_rows:,} Total Rows, {raw_df.shape[1]} Columns")
print("=" * 80)
print("Null counts per required column before filtering:")
print(raw_df[all_required_cols].isnull().sum())
print("-" * 80)

# 3. Strictly Drop Rows With >= 1 Null in Required Features or Target
clean_df = raw_df.dropna(subset=all_required_cols).copy()
clean_rows = len(clean_df)
dropped_rows = initial_rows - clean_rows

print(f"✓ Dropped {dropped_rows:,} rows with missing values (Retained {clean_rows:,} complete cases, {clean_rows/initial_rows*100:.2f}%)")

# 4. Encode 'sex' Feature (M -> 1, F -> 0)
clean_df['sex_encoded'] = clean_df['sex'].astype(str).str.strip().str.upper().map({'M': 1, 'MALE': 1, 'F': 0, 'FEMALE': 0})
clean_df = clean_df.dropna(subset=['sex_encoded']).copy()
clean_df['sex_encoded'] = clean_df['sex_encoded'].astype(int)
clean_df[target_col] = clean_df[target_col].astype(int)

feature_names = ['age', 'sex_encoded', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2']

print("\nCleaned FedMML Target Distribution ('esi_level'):")
esi_dist = clean_df[target_col].value_counts().sort_index()
for cls_val, cnt in esi_dist.items():
    print(f"  * ESI Level {cls_val} : {cnt:,} cases ({cnt/len(clean_df)*100:.2f}%)")
print("=" * 80)

# 5. Stratified 3-Way Partitioning based on config/triage_conf.json
X_all = clean_df[feature_names].values
y_all = clean_df[target_col].values

# (a) Extract Stratified Holdout Test Set
X_rem_raw, X_test_raw, y_rem, y_test = train_test_split(
    X_all, y_all, test_size=test_size, stratify=y_all, random_state=seed_val
)

# (b) Extract Stratified Validation Set from remainder
val_adj_fraction = val_size / (1.0 - test_size)
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X_rem_raw, y_rem, test_size=val_adj_fraction, stratify=y_rem, random_state=seed_val + 1
)

print(f"Stratified Partition Complete (Referenced from triage_conf.json):")
print(f"  * Train Set      : {len(X_train_raw):,} rows ({len(X_train_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Validation Set : {len(X_val_raw):,} rows ({len(X_val_raw)/len(clean_df)*100:.1f}%)")
print(f"  * Holdout Test   : {len(X_test_raw):,} rows ({len(X_test_raw)/len(clean_df)*100:.1f}%)")

In [ ]:
# ---------------------------------------------------------
# Step 2: Feature Matrix Normalization (StandardScaler on Continuous Vitals)
# ---------------------------------------------------------
cont_indices = [0, 2, 3, 4, 5]  # age, systolic_bp, heart_rate, respiratory_rate, spo2

scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val   = X_val_raw.copy()
X_test  = X_test_raw.copy()

X_train[:, cont_indices] = scaler.fit_transform(X_train_raw[:, cont_indices])
X_val[:, cont_indices]   = scaler.transform(X_val_raw[:, cont_indices])
X_test[:, cont_indices]  = scaler.transform(X_test_raw[:, cont_indices])

print(f"✓ Feature Matrices Normalized: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Load External 5V Dataset (5v_cleandf.RData) & Translate Features
# Strictly skip/drop any row with >= 1 NA feature or NA target class
# ---------------------------------------------------------
data_5v_path <- "../datasets/5v_cleandf.RData"
if (!file.exists(data_5v_path)) data_5v_path <- "datasets/5v_cleandf.RData"

env_5v <- new.env()
load(data_5v_path, envir = env_5v)

df_names_5v <- ls(env_5v)[sapply(ls(env_5v), function(x) is.data.frame(get(x, envir = env_5v)))]
df_sizes_5v <- sapply(df_names_5v, function(x) nrow(get(x, envir = env_5v)))
raw_5v_df   <- get(df_names_5v[which.max(df_sizes_5v)], envir = env_5v)

cat("========================================================================\n")
cat(sprintf("  EXTERNAL 5V DATASET LOADED: %d Total Rows, %d Columns\n", nrow(raw_5v_df), ncol(raw_5v_df)))
cat("========================================================================\n")

# Feature translation to match FedMML format:
# - age              -> age
# - gender           -> sex_encoded (Male: 1, Female: 0)
# - triage_vital_sbp -> systolic_bp
# - triage_vital_hr  -> heart_rate
# - triage_vital_rr  -> respiratory_rate
# - triage_vital_o2  -> spo2
# - esi              -> esi_level (1..5)

gender_num_5v <- ifelse(is.na(raw_5v_df$gender), NA, ifelse(as.character(raw_5v_df$gender) == "Male", 1, 0))
esi_num_5v    <- as.numeric(as.character(raw_5v_df$esi))

df_5v_mapped <- data.frame(
  age              = raw_5v_df$age,
  sex_encoded      = gender_num_5v,
  systolic_bp      = raw_5v_df$triage_vital_sbp,
  heart_rate       = raw_5v_df$triage_vital_hr,
  respiratory_rate = raw_5v_df$triage_vital_rr,
  spo2             = raw_5v_df$triage_vital_o2,
  esi_level        = esi_num_5v
)

cat("Null counts per required feature and target before filtering:\n")
print(sapply(df_5v_mapped, function(x) sum(is.na(x))))
cat("------------------------------------------------------------------------\n")

# Strictly drop any row containing >= 1 null value in the 6 features or target
clean_5v_df <- na.omit(df_5v_mapped)
dropped_5v_rows <- nrow(df_5v_mapped) - nrow(clean_5v_df)

cat(sprintf("5V Complete Case Filtering: Dropped %d rows with >= 1 NA value (Retained %d complete cases, %.2f%%)\n", 
            dropped_5v_rows, nrow(clean_5v_df), (nrow(clean_5v_df) / nrow(raw_5v_df)) * 100))
cat("5V Cleaned Target ESI Distribution (100% complete cases):\n")
print(table(clean_5v_df$esi_level))
cat("========================================================================\n")

export_5v_mat <- as.matrix(clean_5v_df)

In [ ]:
# ---------------------------------------------------------
# Step 4: Compute Target Class Distribution & Weights from the 5V Dataset
# ---------------------------------------------------------
from rpy2.robjects import r

mat_5v = np.array(r('export_5v_mat'), dtype=np.float64)
X_5v_raw = mat_5v[:, :6]   # age, sex_encoded, systolic_bp, heart_rate, respiratory_rate, spo2
y_5v     = mat_5v[:, 6].astype(int)  # ESI 1..5

# Verify complete cases
assert not np.isnan(X_5v_raw).any(), "Error: Found NaN values in 5V feature matrix!"
assert not np.isnan(y_5v).any(), "Error: Found NaN values in 5V target vector!"

# Normalize continuous columns using FedMML fitted StandardScaler
X_5v = X_5v_raw.copy()
X_5v[:, cont_indices] = scaler.transform(X_5v_raw[:, cont_indices])

# Compute empirical class frequencies and balanced weights on 5V
classes_1idx = np.array([1, 2, 3, 4, 5])
weights_5v_arr = compute_class_weight(class_weight='balanced', classes=classes_1idx, y=y_5v)
weights_5v_1idx = {int(c): float(w) for c, w in zip(classes_1idx, weights_5v_arr)}
weights_5v_0idx = {int(c - 1): float(w) for c, w in zip(classes_1idx, weights_5v_arr)}

print("=" * 80)
print("  COMPUTED 5V-ADAPTED TARGET CLASS WEIGHTS (BASED ON 282,558 SAMPLES)")
print("=" * 80)
unique_5v, counts_5v = np.unique(y_5v, return_counts=True)
for c, cnt, w in zip(unique_5v, counts_5v, weights_5v_arr):
    print(f"  * ESI {c}: Count = {cnt:,} ({cnt/len(y_5v)*100:6.2f}%) | Weight = {w:8.4f}")
print("=" * 80)

In [ ]:
# ---------------------------------------------------------
# Step 5: Train Models for Both Benchmarks (Standard FedMML Weights vs 5V-Adapted Weights)
# ---------------------------------------------------------
def binary_numpy_smote(X, y_bin, seed=42):
    np.random.seed(seed)
    pos_mask = (y_bin == 1)
    neg_mask = (y_bin == 0)
    n_pos = np.sum(pos_mask)
    n_neg = np.sum(neg_mask)
    if n_pos == 0 or n_neg == 0 or n_pos == n_neg:
        return X, y_bin
    if n_pos < n_neg:
        min_X = X[pos_mask]; target_syn = n_neg - n_pos; min_label = 1
    else:
        min_X = X[neg_mask]; target_syn = n_pos - n_neg; min_label = 0
    n_min = len(min_X)
    syn_X = np.zeros((target_syn, X.shape[1]), dtype=np.float64)
    for i in range(target_syn):
        idx1 = np.random.randint(0, n_min)
        idx2 = np.random.randint(0, n_min)
        alpha = np.random.rand()
        syn_X[i] = min_X[idx1] + alpha * (min_X[idx2] - min_X[idx1])
    return np.vstack([X, syn_X]), np.hstack([y_bin, np.full(target_syn, min_label)])

def compute_hierarchical_probs(l1, l2, l3a, l3b, X_in):
    p1  = l1.predict_proba(X_in)[:, 1]
    p2  = l2.predict_proba(X_in)[:, 1]
    p3a = l3a.predict_proba(X_in)[:, 1]
    p3b = l3b.predict_proba(X_in)[:, 1]
    
    P = np.zeros((len(X_in), 5))
    P[:, 0] = p1
    P[:, 1] = (1 - p1) * p2 * p3a
    P[:, 2] = (1 - p1) * p2 * (1 - p3a)
    P[:, 3] = (1 - p1) * (1 - p2) * p3b
    P[:, 4] = (1 - p1) * (1 - p2) * (1 - p3b)
    return P

# =========================================================
# BENCHMARK 1: STANDARD FEDMML BALANCED WEIGHTING
# =========================================================
print("Training Benchmark 1 Models (Standard FedMML Balanced Weighting)...")

# Model A1: Direct Multiclass LightGBM (Standard)
mc_params_std = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
    'learning_rate': 0.05, 'num_leaves': 31, 'max_depth': 6,
    'class_weight': 'balanced', 'feature_fraction': 0.85, 'bagging_fraction': 0.85,
    'bagging_freq': 1, 'min_child_samples': 20, 'n_estimators': 200,
    'verbosity': -1, 'random_state': 42
}
model_mc_std = lgb.LGBMClassifier(**mc_params_std)
model_mc_std.fit(X_train, y_train - 1, eval_set=[(X_val, y_val - 1)], callbacks=[lgb.early_stopping(15, verbose=False)])

# Model B1: Hierarchical Stacking LightGBM (Standard)
bin_params = {
    'objective': 'binary', 'metric': 'binary_logloss', 'learning_rate': 0.05,
    'num_leaves': 31, 'max_depth': 6, 'feature_fraction': 0.85, 'bagging_fraction': 0.85,
    'bagging_freq': 1, 'verbosity': -1, 'random_state': 42, 'n_estimators': 150
}
X_sm1, y_sm1 = binary_numpy_smote(X_train, (y_train == 1).astype(int))
l1_std = lgb.LGBMClassifier(**bin_params)
l1_std.fit(X_sm1, y_sm1, eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m2_tr = (y_train != 1); m2_val = (y_val != 1)
X_sm2, y_sm2 = binary_numpy_smote(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int))
l2_std = lgb.LGBMClassifier(**bin_params)
l2_std.fit(X_sm2, y_sm2, eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m3a_tr = np.isin(y_train, [2, 3]); m3a_val = np.isin(y_val, [2, 3])
X_sm3a, y_sm3a = binary_numpy_smote(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int))
l3a_std = lgb.LGBMClassifier(**bin_params)
l3a_std.fit(X_sm3a, y_sm3a, eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

m3b_tr = np.isin(y_train, [4, 5]); m3b_val = np.isin(y_val, [4, 5])
X_sm3b, y_sm3b = binary_numpy_smote(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int))
l3b_std = lgb.LGBMClassifier(**bin_params)
l3b_std.fit(X_sm3b, y_sm3b, eval_set=[(X_val[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

val_probs_std = compute_hierarchical_probs(l1_std, l2_std, l3a_std, l3b_std, X_val)
meta_std = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
meta_std.fit(val_probs_std, y_val)
print("✓ Benchmark 1 models successfully trained!")

# =========================================================
# BENCHMARK 2: 5V-ADAPTED CLASS WEIGHTING
# =========================================================
print("\nTraining Benchmark 2 Models (5V-Adapted Class Weighting)...")

# Model A2: Direct Multiclass LightGBM with 5V-Adapted Class Weights
mc_params_5w = mc_params_std.copy()
mc_params_5w['class_weight'] = weights_5v_0idx
model_mc_5w = lgb.LGBMClassifier(**mc_params_5w)
model_mc_5w.fit(X_train, y_train - 1, eval_set=[(X_val, y_val - 1)], callbacks=[lgb.early_stopping(15, verbose=False)])

# Model B2: Hierarchical Stacking LightGBM with 5V-Calibrated Meta-Learner
# Scale pos weights derived from 5V target ratios:
spw_l1 = (len(y_5v) - counts_5v[0]) / max(1, counts_5v[0])  # Non-ESI 1 vs ESI 1
spw_l2 = (counts_5v[3] + counts_5v[4]) / max(1, counts_5v[1] + counts_5v[2])  # 4,5 vs 2,3
spw_l3a = counts_5v[2] / max(1, counts_5v[1])  # ESI 3 vs ESI 2
spw_l3b = counts_5v[4] / max(1, counts_5v[3])  # ESI 5 vs ESI 4

l1_5w = lgb.LGBMClassifier(**bin_params, scale_pos_weight=spw_l1)
l1_5w.fit(X_train, (y_train == 1).astype(int), eval_set=[(X_val, (y_val == 1).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

l2_5w = lgb.LGBMClassifier(**bin_params, scale_pos_weight=spw_l2)
l2_5w.fit(X_train[m2_tr], np.isin(y_train[m2_tr], [2, 3]).astype(int), eval_set=[(X_val[m2_val], np.isin(y_val[m2_val], [2, 3]).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

l3a_5w = lgb.LGBMClassifier(**bin_params, scale_pos_weight=spw_l3a)
l3a_5w.fit(X_train[m3a_tr], (y_train[m3a_tr] == 2).astype(int), eval_set=[(X_val[m3a_val], (y_val[m3a_val] == 2).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

l3b_5w = lgb.LGBMClassifier(**bin_params, scale_pos_weight=spw_l3b)
l3b_5w.fit(X_train[m3b_tr], (y_train[m3b_tr] == 4).astype(int), eval_set=[(X_val[m3b_val], (y_val[m3b_val] == 4).astype(int))], callbacks=[lgb.early_stopping(10, verbose=False)])

val_probs_5w = compute_hierarchical_probs(l1_5w, l2_5w, l3a_5w, l3b_5w, X_val)
meta_5w = LogisticRegression(class_weight=weights_5v_1idx, max_iter=1000, random_state=42)
meta_5w.fit(val_probs_5w, y_val)
print("✓ Benchmark 2 (5V-Adapted Weighting) models successfully trained!")

In [ ]:
# ---------------------------------------------------------
# Step 6: Comprehensive Benchmark Evaluation & Cross-Model Comparison
# ---------------------------------------------------------
def get_per_class_breakdown(y_true, y_pred, probs, pipeline_name, dataset_name, class_list=[1, 2, 3, 4, 5]):
    rows = []
    recalls, specs, bal_accs, precs, f1s, aucs = [], [], [], [], [], []
    for idx, cls in enumerate(class_list):
        y_bin_true = (y_true == cls).astype(int)
        y_bin_pred = (y_pred == cls).astype(int)
        tp = np.sum((y_bin_true == 1) & (y_bin_pred == 1))
        fn = np.sum((y_bin_true == 1) & (y_bin_pred == 0))
        fp = np.sum((y_bin_true == 0) & (y_bin_pred == 1))
        tn = np.sum((y_bin_true == 0) & (y_bin_pred == 0))
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        bal  = (rec + spec) / 2.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        f1   = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        try: auc = roc_auc_score(y_bin_true, probs[:, idx])
        except Exception: auc = 0.0
        
        recalls.append(rec); specs.append(spec); bal_accs.append(bal)
        precs.append(prec); f1s.append(f1); aucs.append(auc)
        
        rows.append({
            'Dataset': dataset_name,
            'Pipeline': pipeline_name,
            'Class': f'ESI_{cls}',
            'Recall_Sensitivity': round(rec, 4),
            'Specificity': round(spec, 4),
            'Balanced_Accuracy': round(bal, 4),
            'Precision_PPV': round(prec, 4),
            'F1_Score': round(f1, 4),
            'ROC_AUC': round(auc, 4)
        })
    rows.append({
        'Dataset': dataset_name,
        'Pipeline': pipeline_name,
        'Class': 'Macro_Average',
        'Recall_Sensitivity': round(np.mean(recalls), 4),
        'Specificity': round(np.mean(specs), 4),
        'Balanced_Accuracy': round(np.mean(bal_accs), 4),
        'Precision_PPV': round(np.mean(precs), 4),
        'F1_Score': round(np.mean(f1s), 4),
        'ROC_AUC': round(np.mean(aucs), 4)
    })
    return pd.DataFrame(rows)

# 1. Predictions on External 5V Dataset (N = 282,558)
# (a) Standard Models on 5V
p_5v_mc_std  = model_mc_std.predict_proba(X_5v)
y_5v_mc_std  = model_mc_std.predict(X_5v) + 1
r_5v_mc_std  = get_per_class_breakdown(y_5v, y_5v_mc_std, p_5v_mc_std, 'Direct_Multiclass_Standard', 'External_5V')

p_5v_st_raw_std = compute_hierarchical_probs(l1_std, l2_std, l3a_std, l3b_std, X_5v)
p_5v_st_std     = meta_std.predict_proba(p_5v_st_raw_std)
y_5v_st_std     = meta_std.predict(p_5v_st_raw_std)
r_5v_st_std     = get_per_class_breakdown(y_5v, y_5v_st_std, p_5v_st_std, 'Hierarchical_Stacking_Standard', 'External_5V')

# (b) 5V-Adapted Models on 5V
p_5v_mc_5w  = model_mc_5w.predict_proba(X_5v)
y_5v_mc_5w  = model_mc_5w.predict(X_5v) + 1
r_5v_mc_5w  = get_per_class_breakdown(y_5v, y_5v_mc_5w, p_5v_mc_5w, 'Direct_Multiclass_5V_Adapted', 'External_5V')

p_5v_st_raw_5w = compute_hierarchical_probs(l1_5w, l2_5w, l3a_5w, l3b_5w, X_5v)
p_5v_st_5w     = meta_5w.predict_proba(p_5v_st_raw_5w)
y_5v_st_5w     = meta_5w.predict(p_5v_st_raw_5w)
r_5v_st_5w     = get_per_class_breakdown(y_5v, y_5v_st_5w, p_5v_st_5w, 'Hierarchical_Stacking_5V_Adapted', 'External_5V')

print("=" * 115)
print("   EXTERNAL 5V GENERALIZATION BENCHMARK: STANDARD vs 5V-ADAPTED CLASS WEIGHTING (N = 282,558)")
print("=" * 115)

comp_5v_rows = []
for i in range(len(r_5v_mc_std)):
    cls_lbl = r_5v_mc_std.loc[i, 'Class']
    b_mc_std = r_5v_mc_std.loc[i, 'Balanced_Accuracy']
    b_mc_5w  = r_5v_mc_5w.loc[i, 'Balanced_Accuracy']
    b_st_std = r_5v_st_std.loc[i, 'Balanced_Accuracy']
    b_st_5w  = r_5v_st_5w.loc[i, 'Balanced_Accuracy']
    
    comp_5v_rows.append({
        'Class': cls_lbl,
        'MC_Std_BalAcc': f"{b_mc_std*100:.2f}%",
        'MC_5V_Adapted_BalAcc': f"{b_mc_5w*100:.2f}%",
        'Delta_MC': f"{(b_mc_5w - b_mc_std)*100:+.2f}%",
        'Stack_Std_BalAcc': f"{b_st_std*100:.2f}%",
        'Stack_5V_Adapted_BalAcc': f"{b_st_5w*100:.2f}%",
        'Delta_Stack': f"{(b_st_5w - b_st_std)*100:+.2f}%"
    })

comp_5v_df = pd.DataFrame(comp_5v_rows)
print(comp_5v_df.to_string(index=False))
print("=" * 115 + chr(10))

# Export CSV Reports
reports_dir = f"{ROOT}/reports"
os.makedirs(reports_dir, exist_ok=True)

all_reports_5v = pd.concat([r_5v_mc_std, r_5v_st_std, r_5v_mc_5w, r_5v_st_5w], ignore_index=True)
all_reports_5v.to_csv(os.path.join(reports_dir, 'external_5v_all_models_detailed_report.csv'), index=False)
comp_5v_df.to_csv(os.path.join(reports_dir, 'external_5v_weight_adaptation_comparison.csv'), index=False)
print(f"✓ Detailed benchmark comparisons exported to {reports_dir}/")

In [ ]:
# ---------------------------------------------------------
# Step 7: Diagnostic Visualizations (2x2 Confusion Matrix Comparison on 5V Dataset)
# ---------------------------------------------------------
plots_dir = f"{ROOT}/plots"
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

esi_labels = [f"ESI {i}" for i in range(1, 6)]

fig, axes = plt.subplots(2, 2, figsize=(18, 15))

def render_cm(ax, y_true, y_pred, title_text, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred, labels=[1, 2, 3, 4, 5])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    annot = np.empty_like(cm, dtype=object)
    for i in range(5):
        for j in range(5):
            annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)"
    sns.heatmap(cm_norm, annot=annot, fmt='', cmap=cmap, cbar=True, ax=ax,
                vmin=0, vmax=1, xticklabels=esi_labels, yticklabels=esi_labels)
    ax.set_title(title_text, fontsize=11.5, fontweight='bold', pad=10)
    ax.set_xlabel("Predicted ESI Level", fontsize=10.5, fontweight='bold')
    ax.set_ylabel("True ESI Level", fontsize=10.5, fontweight='bold')

render_cm(axes[0, 0], y_5v, y_5v_mc_std, 
          f"[5V Dataset] Direct Multiclass LightGBM (Standard FedMML Weights)\nMacro Bal Acc: {r_5v_mc_std.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {r_5v_mc_std.loc[5, 'ROC_AUC']:.4f}", cmap='Blues')

render_cm(axes[0, 1], y_5v, y_5v_mc_5w, 
          f"[5V Dataset] Direct Multiclass LightGBM (5V-Adapted Class Weights)\nMacro Bal Acc: {r_5v_mc_5w.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {r_5v_mc_5w.loc[5, 'ROC_AUC']:.4f}", cmap='Oranges')

render_cm(axes[1, 0], y_5v, y_5v_st_std, 
          f"[5V Dataset] Hierarchical Stacking LightGBM (Standard FedMML Weights)\nMacro Bal Acc: {r_5v_st_std.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {r_5v_st_std.loc[5, 'ROC_AUC']:.4f}", cmap='Greens')

render_cm(axes[1, 1], y_5v, y_5v_st_5w, 
          f"[5V Dataset] Hierarchical Stacking LightGBM (5V-Adapted Class Weights)\nMacro Bal Acc: {r_5v_st_5w.loc[5, 'Balanced_Accuracy']*100:.2f}% | AUC: {r_5v_st_5w.loc[5, 'ROC_AUC']:.4f}", cmap='Purples')

plt.tight_layout()
cm_grid_path = os.path.join(plots_dir, "external_5v_weight_adaptation_confusion_matrix_grid.png")
plt.savefig(cm_grid_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'external_5v_weight_adaptation_confusion_matrix_grid.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Confusion Matrix comparison grid saved to: {cm_grid_path}")

In [ ]:
# ---------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------
deploy_dir = f"{ROOT}/deploy"
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'scaler': scaler,
    'feature_names': feature_names,
    'model_mc_std': model_mc_std,
    'model_mc_5w': model_mc_5w,
    'hierarchical_std': {
        'l1': l1_std, 'l2': l2_std, 'l3a': l3a_std, 'l3b': l3b_std, 'meta': meta_std
    },
    'hierarchical_5w': {
        'l1': l1_5w, 'l2': l2_5w, 'l3a': l3a_5w, 'l3b': l3b_5w, 'meta': meta_5w
    },
    'weights_5v_1idx': weights_5v_1idx,
    'weights_5v_0idx': weights_5v_0idx,
    'config_training': config['training']
}

bundle_file = os.path.join(deploy_dir, 'fedmml_lightgbm_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='FedMML_LightGBM_ED_Triage_Classifier_With_5V_Class_Weight_Adaptation',
    training_dataset='datasets/fedmml_ed_triage_dataset.csv',
    external_dataset='datasets/5v_cleandf.RData',
    config_file='config/triage_conf.json',
    config_training=config['training'],
    features=['age', 'sex', 'systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2'],
    features_5v_mapping={
        'age': 'age',
        'sex': 'gender',
        'systolic_bp': 'triage_vital_sbp',
        'heart_rate': 'triage_vital_hr',
        'respiratory_rate': 'triage_vital_rr',
        'spo2': 'triage_vital_o2',
        'esi_level': 'esi'
    },
    weights_5v=weights_5v_1idx,
    fedmml_complete_cases=len(clean_df),
    ext_5v_complete_cases=len(mat_5v),
    external_5v_comparison=comp_5v_rows
)

manifest_file = os.path.join(deploy_dir, 'fedmml_lightgbm_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Production Bundle   : {bundle_file}")
print(f"✓ Production Manifest : {manifest_file}")